# 시군구 코드 매핑 — 법정동 → KOSIS

V-World shapefile에서 쓰는 **법정동코드**(EMD 앞 5자리)랑 KOSIS **행정구역코드**는 같은 지역이어도 번호가 달라서 변환 테이블이 필요.  
행안부 `KiKcd_B`에서 시도/시군구 이름 뽑고, KOSIS 고령화율 CSV랑 이름 기준으로 매칭.

**출력:** `legal_to_kosis_sgg.csv`


## 1. 설정

In [1]:
import pandas as pd

JSCODE_CSV = '시군구_맞추기용/KIKcd_B.20260325.csv'
AGING_CSV = 'aging_rate_2019_2023.csv'
OUTPUT_CSV = 'legal_to_kosis_sgg.csv'

# jscode 시도명 → KOSIS 시도코드(앞 2자리)
SIDO_TO_KOSIS = {
    '서울특별시': '11', '부산광역시': '21', '대구광역시': '22', '인천광역시': '23',
    '광주광역시': '24', '대전광역시': '25', '울산광역시': '26', '세종특별자치시': '29',
    '경기도': '31', '강원특별자치도': '32', '강원도': '32', '충청북도': '33',
    '충청남도': '34', '전북특별자치도': '35', '전라북도': '35', '전라남도': '36',
    '경상북도': '37', '경상남도': '38', '제주특별자치도': '39',
}

## 2. jscode 법정동 → 시군구명

In [2]:
df_b = pd.read_csv(JSCODE_CSV, encoding='utf-8-sig', dtype=str)
# 현재 유효한 코드만 (말소된 행 제외)
df_b = df_b[df_b['말소일자'].isna() | (df_b['말소일자'] == 'nan')]

rows = []
emd = df_b[df_b['읍면동명'].notna()].copy()
emd['legal5'] = emd['법정동코드'].str[:5]
for legal5, g in emd.groupby('legal5'):
    rows.append({
        'legal5': legal5,
        '시도명': g['시도명'].iloc[0],
        '시군구명_법정': g['시군구명'].iloc[0],
    })

existing = {r['legal5'] for r in rows}
sgg_only = df_b[df_b['읍면동명'].isna() & df_b['시군구명'].notna()].copy()
sgg_only['legal5'] = sgg_only['법정동코드'].str[:5]
for _, r in sgg_only.iterrows():
    if r['legal5'] not in existing:
        rows.append({
            'legal5': r['legal5'],
            '시도명': r['시도명'],
            '시군구명_법정': r['시군구명'],
        })

legal_df = pd.DataFrame(rows).drop_duplicates('legal5')
print(f'법정 시군구 코드: {len(legal_df)}개')
legal_df.head()

법정 시군구 코드: 272개


,legal5,시도명,시군구명_법정
0,11110,서울특별시,종로구
1,11140,서울특별시,중구
2,11170,서울특별시,용산구
3,11200,서울특별시,성동구
4,11215,서울특별시,광진구


## 3. KOSIS 시군구명과 매칭

In [ ]:
def kosis_match_names(full_name):
    if pd.isna(full_name) or not str(full_name).strip():
        return []
    name = str(full_name).strip()
    candidates = [name]
    if ' ' in name:
        parts = name.split()
        candidates.append(parts[-1])
        candidates.append(parts[0])
    seen = set()
    return [c for c in candidates if not (c in seen or seen.add(c))]


aging = pd.read_csv(AGING_CSV)
aging_sgg = aging[['SGG_CODE', 'SGG_NM']].drop_duplicates()
aging_sgg['SGG_CODE'] = aging_sgg['SGG_CODE'].astype(str).str.zfill(5)
aging_sgg['kosis_sido'] = aging_sgg['SGG_CODE'].str[:2]

lookup = {
    (r['kosis_sido'], r['SGG_NM']): r['SGG_CODE']
    for _, r in aging_sgg.iterrows()
}

results = []
for _, r in legal_df.iterrows():
    sido = r['시도명']
    sgg_name = r['시군구명_법정']
    kosis_sido = SIDO_TO_KOSIS.get(sido)
    kosis_code = None
    kosis_nm = None

    if sido == '세종특별자치시' and (pd.isna(sgg_name) or not str(sgg_name).strip()):
        kosis_code = lookup.get(('29', '세종시'))
        kosis_nm = '세종시'
    elif kosis_sido:
        for cand in kosis_match_names(sgg_name):
            code = lookup.get((kosis_sido, cand))
            if code:
                kosis_code = code
                kosis_nm = cand
                break

    results.append({
        'legal5': r['legal5'],
        '시도명': sido,
        '시군구명_법정': sgg_name,
        'KOSIS_SGG_CODE': kosis_code,
        'KOSIS_SGG_NM': kosis_nm,
    })

mapping = pd.DataFrame(results)
matched = mapping['KOSIS_SGG_CODE'].notna().sum()
print(f'매칭 성공: {matched} / {len(mapping)}')
if matched < len(mapping):
    print('\n매칭 실패:')
    print(mapping[mapping['KOSIS_SGG_CODE'].isna()])
mapping.head(10)

매칭 성공: 267 / 272

매칭 실패:
    legal5    시도명 시군구명_법정 KOSIS_SGG_CODE KOSIS_SGG_NM
255  28115  인천광역시  중구영종출장           None         None
256  28116  인천광역시  중구용유출장           None         None
257  28265  인천광역시  서구검단출장           None         None
269  48120   경상남도     창원시           None         None
270  48245   경상남도  사천남양출장           None         None


,legal5,시도명,시군구명_법정,KOSIS_SGG_CODE,KOSIS_SGG_NM
0,11110,서울특별시,종로구,11010,종로구
1,11140,서울특별시,중구,11020,중구
2,11170,서울특별시,용산구,11030,용산구
3,11200,서울특별시,성동구,11040,성동구
4,11215,서울특별시,광진구,11050,광진구
5,11230,서울특별시,동대문구,11060,동대문구
6,11260,서울특별시,중랑구,11070,중랑구
7,11290,서울특별시,성북구,11080,성북구
8,11305,서울특별시,강북구,11090,강북구
9,11320,서울특별시,도봉구,11100,도봉구


## 4. 저장 및 검증

In [4]:
mapping.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_CSV}')

# 종로구 예시: 법정 11110 → KOSIS 11010
sample = mapping[mapping['legal5'] == '11110'][
    ['legal5', '시군구명_법정', 'KOSIS_SGG_CODE', 'KOSIS_SGG_NM']
]
print('\n=== 검증: 종로구 ===')
print(sample.to_string(index=False))

저장 완료: legal_to_kosis_sgg.csv

=== 검증: 종로구 ===
legal5 시군구명_법정 KOSIS_SGG_CODE KOSIS_SGG_NM
 11110     종로구          11010          종로구
